# 第 2 章 · 分词器与数据流水线

> 📄 配套动画网页：`../02_tokenizer.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第一部分 · 基础

模型看不见文字，只能看见整数。这一章我们先把「文字 ↔ 整数」的桥梁搭好，再把数据组织成模型能吃的 batch。

---


## 🎯 本章目标

- 理解 BPE 子词分词的核心思想
- 会用 `collections.Counter` 做统计、会手写一次 BPE 合并
- 掌握 `tokenizers` 库训练一个小词表，以及 `AutoTokenizer` 的用法
- 会用 `torch.utils.data.Dataset` + `DataLoader` 组装训练数据

---


## 📖 背景：为什么是子词（subword）

- 按**字符**切：序列太长，模型要记很久才能到下一个词。
- 按**词**切：词表爆炸，遇到没见过的词就崩（OOV）。
- 按**子词**切：折中。常见片段合成一个 token，罕见词拆成小片段。

MiniMind 用 **BPE**，词表只有 **6400**。此外还有几个特殊 token 只做结构标记：
`<|im_start|>` / `<|im_end|>`（对话边界）、`<think>`（思维链）、`<tool_call>`（工具调用）。

---



## 🖼 官方配图

![MiniMind 各阶段数据集的构成——分词质量直接决定这些数据能被模型利用多少](../images/dataset.jpg)

*MiniMind 各阶段数据集的构成——分词质量直接决定这些数据能被模型利用多少*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 2.1 字符与编码：`ord` / `chr` / `encode`

分词的最底层就是字节和字符。

In [ ]:
s = "猫"
print("字符     :", s)
print("Unicode  :", hex(ord(s)))
print("UTF-8 字节:", list(s.encode("utf-8")))
print("转回字符 :", chr(ord(s)))

# 中文一个字通常占 3 个 UTF-8 字节
print("字节长度 :", len(s.encode("utf-8")))

### 2.2 统计频次：`collections.Counter`

BPE 的第一步就是数「哪个符号对出现得最多」。

In [ ]:
from collections import Counter

words = ["low", "lower", "newest", "widest", "low"]
cnt = Counter(words)
print("词频:", dict(cnt))
print("最常见的 2 个:", cnt.most_common(2))

# 统计“相邻字符对”的频次
pairs = Counter()
for w, n in cnt.items():
    for i in range(len(w) - 1):
        pairs[w[i:i+2]] += n
print("相邻对频次:", dict(pairs))
print("最频繁的相邻对:", pairs.most_common(1))

### 2.3 手写一次 BPE 合并

BPE 的规则只有一句：**反复把出现次数最多的相邻符号对合并成一个新符号**。

In [ ]:
def merge_pair(word_list, a, b):
    """把 word_list 中所有相邻的 (a, b) 合并成 a+b"""
    merged = a + b
    out = []
    for w in word_list:
        new_w, i = [], 0
        while i < len(w):
            if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                new_w.append(merged); i += 2
            else:
                new_w.append(w[i]); i += 1
        out.append(new_w)
    return out

# 把每个词拆成字符
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
words = [list(w) for w in corpus]
print("初始:", words)

words = merge_pair(words, "e", "s")     # 合并 e+s → es
print("合并 es:", words)
words = merge_pair(words, "es", "t")    # 再合并 es+t → est
print("合并 est:", words)

### 2.4 用 `tokenizers` 训练一个小 BPE

真实项目不会手写 BPE，而是用 `tokenizers` 库。下面在一个小语料上训练一个迷你词表。

In [ ]:
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
    HAS_TOKENIZERS = True
except ImportError:
    HAS_TOKENIZERS = False
    print("未安装 tokenizers —— 请先在项目目录执行 `uv sync`，然后重启内核。")

if HAS_TOKENIZERS:
    tok = Tokenizer(BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=120,
        special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    )
    corpus_text = [
        "minimind is a tiny language model",
        "train a language model from scratch",
        "the cat sat on the mat",
        "a small model can still learn to chat",
    ]
    tok.train_from_iterator(corpus_text, trainer=trainer)

    enc = tok.encode("minimind is a tiny model")
    print("词表大小:", tok.get_vocab_size())
    print("tokens  :", enc.tokens)
    print("ids     :", enc.ids)

### 2.5 加载 MiniMind 官方分词器（可选）

如果已经下载了 MiniMind 的 `tokenizer.json`，可以直接加载。否则跳过这一格。

In [ ]:
from pathlib import Path

# 官方分词器已放在 ../data/ 下，可直接使用
tok_path = Path("../data/tokenizer.json")

if not tok_path.exists():
    print("未找到 ../data/tokenizer.json —— 跳过。可从 MiniMind 仓库 model/ 目录复制过来。")
else:
    loaded = False
    try:
        from transformers import AutoTokenizer
        mm_tok = AutoTokenizer.from_pretrained("../data")
        ids = mm_tok("你好，MiniMind！").input_ids
        print("MiniMind 词表大小:", mm_tok.vocab_size)
        print("编码:", ids)
        print("解码:", mm_tok.decode(ids))
        loaded = True
    except ImportError:
        pass

    if not loaded:
        try:
            from tokenizers import Tokenizer
            mm_tok = Tokenizer.from_file(str(tok_path))
            enc = mm_tok.encode("你好，MiniMind！")
            print("MiniMind 词表大小:", mm_tok.get_vocab_size())
            print("tokens:", enc.tokens)
            print("ids   :", enc.ids)
            loaded = True
        except ImportError:
            print("未安装 transformers / tokenizers —— 请先执行 `uv sync` 再重启内核。")

### 2.6 数据流水线：`Dataset` 与 `DataLoader`

`Dataset` 负责「第 i 个样本长什么样」，`DataLoader` 负责「怎么凑成一个 batch」。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    """把一段文本切成定长片段，做下一词预测"""
    def __init__(self, ids, max_length=8):
        self.ids = ids
        self.max_length = max_length

    def __len__(self):
        return len(self.ids) // self.max_length

    def __getitem__(self, index):
        chunk = self.ids[index * self.max_length:(index + 1) * self.max_length]
        input_ids = torch.tensor(chunk, dtype=torch.long)
        labels = input_ids.clone()          # 先复制，稍后错位
        return input_ids, labels

ids = list(range(1, 41))
ds = ToyDataset(ids, max_length=8)
loader = DataLoader(ds, batch_size=2, shuffle=False)
for x, y in loader:
    print("input:", x.tolist())
    print("label:", y.tolist())
    break

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

本章的核心产出是两个东西：一个自己写的 BPE 合并循环，一个 `PretrainDataset`。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 2.7 从零实现 `train_bpe`

给定一个语料（`dict[str, int]`，词 → 频次）和目标合并次数 `num_merges`，
返回合并规则列表，并把语料逐步合并。

**算法**：
1. 把每个词拆成字符列表；
2. 统计所有相邻符号对的加权频次；
3. 取频次最高的对 (a, b)，合并成 a+b，记录规则；
4. 重复 num_merges 次。

In [ ]:
# ======== 你的实现：train_bpe ========
#
# 【要求】
#   - 输入：corpus(dict 词→频次)、num_merges(int)
#   - 输出：merges 列表 [(a, b), ...]
#   - 每一步都要重新统计相邻对频次（因为合并后符号变了）
#   - 频次相同时取字典序较小的一对，保证结果可复现
#
# 【可用到的函数 / 方法】
#   - `list(word)` 把字符串拆成字符
#   - `collections.Counter`、`.most_common()`
#   - 上面写过的 `merge_pair` 可以直接复用
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：train_bpe ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
    merges = train_bpe(corpus, 3)
    assert len(merges) == 3, f"应返回 3 条合并规则，得到 {len(merges)}"
    assert merges[0] == ("e", "s"), f"第一步应合并出现 9 次的 ('e', 's')，得到 {merges[0]}"
    assert merges[1] == ("es", "t"), f"第二步应合并出现 9 次的 ('es', 't')，得到 {merges[1]}"

check("train_bpe 合并顺序与频次统计", _t)

### 2.8 实现 `PretrainDataset`

参考 MiniMind 的写法：读一批文本 → 分词 → 加 BOS/EOS → 补齐到 `max_length` →
**把 pad 位置的标签设为 `-100`**（这样交叉熵会忽略它们）。

**为什么是 -100**：`F.cross_entropy(..., ignore_index=-100)` 会跳过标签为 -100 的位置。

In [ ]:
# ======== 你的实现：PretrainDataset ========
#
# 【要求】
#   - 继承 torch.utils.data.Dataset
#   - __init__ 接收 data_path / tokenizer / max_length（用 json 逐行读 jsonl 即可）
#   - __getitem__ 返回 (input_ids, labels)，两者都是 torch.long 张量
#   - 序列结构：[BOS] + tokens + [EOS] + [PAD] * 补齐长度
#   - labels = input_ids.clone()，然后把 pad 位置改成 -100
#
# 【可用到的函数 / 方法】
#   - `torch.tensor(x, dtype=torch.long)`
#   - `tokenizer(...).input_ids`
#   - `labels[input_ids == pad_id] = -100`
#   - `json.loads(line)` 逐行读取
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：PretrainDataset ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    import json, tempfile, os

    class FakeTok:
        bos_token_id, eos_token_id, pad_token_id = 1, 2, 0
        def __call__(self, text, add_special_tokens=False, max_length=100, truncation=True):
            ids = [3 + (ord(ch) % 50) for ch in str(text)][:max_length]
            return type("Enc", (), {"input_ids": ids})()

    path = os.path.join(tempfile.gettempdir(), "mm_pretrain_ds_test.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps({"text": "hello world"}) + "\n")

    ds = PretrainDataset(path, FakeTok(), max_length=16)
    x, y = ds[0]
    assert tuple(x.shape) == (16,), f"input_ids 形状应为 (16,)，得到 {tuple(x.shape)}"
    assert tuple(y.shape) == (16,), f"labels 形状应为 (16,)，得到 {tuple(y.shape)}"
    assert x[0].item() == 1, "序列应以 BOS(1) 开头"
    assert x[1].item() != 0, "第 2 个位置应是真实 token，不该是 pad"
    assert (y[x == 0] == -100).all(), "pad 位置的标签应全部为 -100"
    assert (y[x != 0] == x[x != 0]).all(), "非 pad 位置的标签应等于 input_ids 本身"

check("PretrainDataset 形状 / BOS / pad 标签", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能用自己的话说清 BPE 的合并规则
- [ ] `train_bpe` 在小语料上跑出合理结果了吗？
- [ ] `PretrainDataset` 返回的两个张量形状一致吗？pad 位置标签是 -100 吗？
- [ ] 用 `DataLoader` 取一个 batch，形状是 `[batch, max_length]` 吗？

**思考题**：如果词表太小，会发生什么？太大呢？MiniMind 为什么选 6400 这个数量级？
